# TC_08: FCNN with hyperparameter variation
Change models hyperparameter (learning rate:0.01, 0.0001 and batch size:32, 128). Run full diagnostic pipeline at each level and compare against TC_03 (Golden Baseline)

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_fcnn, evaluate_model_fcnn
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_tab, mc_dropout_prediction_tab
from src.inference_diagnostics.explainability import shap_tab, lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.utils.config_loader import load_config, get_tabular_config, load_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id

config = load_config()
tracker = PerformanceTracker()

### Load clean data


In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
quality_report = check_tabular_quality(X_train, y_train, config)

# Load seed
seed = config['random_seeds']['primary_seed']
learning_rates =  config['stage2_training_stability']['hyperparameter_variation']['learning_rates']
batch_sizes =  config['stage2_training_stability']['hyperparameter_variation']['batch_sizes']

# Keep original hyperparameters from config
original_lr = config['models']['fcnn']['params']['learning_rate']
original_bs = config['models']['fcnn']['params']['batch_size']

# Identity for this notebook
dataset_short = get_tabular_config(config)['short_name']
model_name = 'fcnn'
test_case = 'TC08'

# Golden baseline thresholds written by TC_03
mc_threshold = load_threshold(config, dataset_short, model_name, 'mc')
de_threshold = load_threshold(config, dataset_short, model_name, 'de')
print(f"Loaded thresholds - MC: {mc_threshold}, DE: {de_threshold}")


Loading dataset.
Dataset: Diabetes Health Indicators
Duplicate rows kept (24206 found).
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators
Loaded thresholds - MC: 0.0616, DE: 0.0302


### Run full pipeline for each perturbation levels

In [3]:
all_results = {}

for lr in learning_rates:
    for bs in batch_sizes:
        print(f"Hyperparameter variations LR: {lr}, BS: {bs}")

        tracker = PerformanceTracker()

        # Change hyperparameters in config
        config['models']['fcnn']['params']['learning_rate'] = lr
        config['models']['fcnn']['params']['batch_size'] = bs

        # Train FCNN with new hyperparameters
        tracker.start_performance_track()
        fcnn_model = train_fcnn(X_train, y_train, config)
        tracker.stop_performance_track(f"FCNN training with hyperparameter variations LR{lr} BS{bs}")

        tracker.start_performance_track()
        fcnn_accuracy, fcnn_prediction, fcnn_report = evaluate_model_fcnn(fcnn_model, X_test, y_test, config)
        tracker.stop_performance_track(f"FCNN evaluation with hyperparameter variations LR{lr} BS{bs}")

        # MC Dropout
        tracker.start_performance_track()
        mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_tab(fcnn_model, X_test, config)
        tracker.stop_performance_track(f"FCNN MC Dropout training with hyperparameter variations LR{lr} BS{bs}")

        print(f"MC Dropout uncertainty status:")
        print(f"Mean: {mc_uncertainty.mean()}")
        print(f"Max: {mc_uncertainty.max()}")
        print(f" 90th percentile (threshold): {mc_threshold}")

        plot_uncertainty_distribution(mc_uncertainty, f"FCNN training with hyperparameter variations LR{lr} BS{bs} ", mc_threshold, config)

        # Deep Ensemble
        tracker.start_performance_track()
        de_means_probabilities, de_uncertainty = deep_ensemble_prediction_tab(train_fcnn, X_train, y_train, X_test, config)
        tracker.stop_performance_track(f"FCNN Deep Ensemble training with hyperparameter variations LR{lr} BS{bs}")

        print(f"Deep Ensembl uncertainty status:")
        print(f"Mean: {de_uncertainty.mean()}")
        print(f"Max: {de_uncertainty.max()}")
        print(f" 90th percentile (threshold): {de_threshold}")

        plot_uncertainty_distribution(de_uncertainty, f"FCNN training with hyperparameter variations LR{lr} BS{bs}", de_threshold, config)

        # SHAP
        tracker.start_performance_track()
        shap_values, shap_samples = shap_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
        tracker.stop_performance_track(f"FCNN SHAP with hyperparameter variations  BS{bs}")
        print(f"SHAP values shape: {shap_values.shape}")

        # LIME
        tracker.start_performance_track()
        lime_explanations, lime_samples = lime_tab(fcnn_model, X_train, X_test, config, is_pytorch = True)
        tracker.stop_performance_track(f"FCNN LIME with hyperparameter variations LR{lr} BS{bs}")
        print(f"LIME explanations: {len(lime_explanations)}")

        # Calculate consistency
        feature_names = list(X_train.columns)
        consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)

        # MC Dropout with XAI flags
        mc_y_predictions = mc_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
        mc_flags = assign_flags(mc_uncertainty[:len(consistency_scores)], consistency_scores, mc_threshold, 0.5)
        mc_flag_results = evaluate_flags(mc_flags, mc_y_predictions, y_test[:len(consistency_scores)])
        plot_flag_distribution(mc_flags, f"FCNN training with hyperparameter variations LR{lr} BS{bs}", config)

        # Deep Ensemble flags
        de_y_predictions = de_means_probabilities[:len(consistency_scores)].argmax(axis = 1)
        de_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, de_threshold, 0.5)
        de_flag_results = evaluate_flags(de_flags, de_y_predictions, y_test[:len(consistency_scores)])
        plot_flag_distribution(de_flags, f"FCNN training with hyperparameter variations LR{lr} BS{bs}", config)

        # MC Dropout without XAI (constant consistency)
        con_consistency = np.ones(len(consistency_scores))
        mc_flags_uq_only = assign_flags(mc_uncertainty[:len(consistency_scores)], con_consistency, mc_threshold, 0.5)

        print(f"MC Dropout without XAI Flagging:")
        mc_only_flag_results = evaluate_flags(mc_flags_uq_only, mc_y_predictions, y_test[:len(consistency_scores)])
        plot_flag_distribution(mc_flags_uq_only, f"FCNN training with hyperparameter variations LR{lr} BS{bs}", config)

        print(f"MC Dropout with XAI green accuracy: {mc_flag_results['GREEN']['accuracy']}")
        print(f"MC Dropout without XAI green accuracy: {mc_only_flag_results['GREEN']['accuracy']}")
        print(f"Improvement with XAI: {mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy']}")

        # Deep Ensemble without XAI (constant consistency)
        de_flags_uq_only = assign_flags(de_uncertainty[:len(consistency_scores)], con_consistency, de_threshold, 0.5)

        print(f"Deep Ensemble without XAI Flagging:")
        de_only_flag_results = evaluate_flags(de_flags_uq_only, de_y_predictions, y_test[:len(consistency_scores)])
        plot_flag_distribution(de_flags_uq_only, f"FCNN training with hyperparameter variations LR{lr} BS{bs}", config)

        print(f"Deep Ensembles with XAI green accuracy: {de_flag_results['GREEN']['accuracy']}")
        print(f"Deep Ensembles without XAI green accuracy: {de_only_flag_results['GREEN']['accuracy']}")
        print(f"Improvement with XAI: {de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy']}")

        level_result = {
        'perturbation': 'hyperparameter variations',
        'learning_rate': lr,
        'batch_size': bs,
        'model': 'FCNN',
        'accuracy': round(fcnn_accuracy, 4),
        'classification_report': fcnn_report,
        'quality_report': quality_report['feature_stats'],
        'mc_uncertainty': {
            'mean': round(float(mc_uncertainty.mean()), 8),
            'max': round(float(mc_uncertainty.max()), 8),
            'threshold': mc_threshold
        },
        'de_uncertainty': {
            'mean': round(float(de_uncertainty.mean()), 8),
            'max': round(float(de_uncertainty.max()), 8),
            'threshold': de_threshold
        },
        'consistency':{
            'mean': round(float(consistency_scores.mean()), 4),
            'min': round(float(consistency_scores.min()), 4),
            'max': round(float(consistency_scores.max()), 4)
        },
        'flagging_mc': mc_flag_results,
        'flagging_de': de_flag_results,
        'flagging_mc_only': mc_only_flag_results,
        'mc_vs_mc_plus_xai': round(mc_flag_results['GREEN']['accuracy'] - mc_only_flag_results['GREEN']['accuracy'], 4),
        'flagging_de_only': de_only_flag_results,
        'de_vs_de_plus_xai': round(de_flag_results['GREEN']['accuracy'] - de_only_flag_results['GREEN']['accuracy'], 4),
        'performance': tracker.get_results()
        }

        all_results[f" LR{lr} BS{bs}"] = level_result

        # Per sample results for this hyperparameter combo
        experiment_id = build_experiment_id(dataset_short, model_name, test_case, f"lr{lr}_bs{bs}")
        save_per_sample(
            config,
            experiment_id,
            y_true=y_test,
            y_pred=mc_y_predictions,
            mc_uncertainty=mc_uncertainty,
            de_uncertainty=de_uncertainty,
            consistency=consistency_scores,
        )

        # Save individual report
        report_output = generate_report(config, f"{get_tabular_config(config)['name']} - FCNN hyperparameter LR{lr} BS{bs}", stage2_result=level_result)
        save_report(report_output, f'{dataset_short.upper()}_TC_08_FCNN_Hyperparameter_LR_{lr}_BS_{bs}.json', config)

# Restore config file
config['models']['fcnn']['params']['learning_rate'] = original_lr
config['models']['fcnn']['params']['batch_size'] = original_bs




Hyperparameter variations LR: 0.01, BS: 32
FCNN training started.
 Early stopping at epoch 6
FCNN training completed.
FCNN training with hyperparameter variations LR0.01 BS32: Time:56.07s, Memory:598.70MB
Model evaluation started for FCNN.
{'0': {'precision': 0.8606709239987386, 'recall': 1.0, 'f1-score': 0.9251189051195406, 'support': 43667.0}, '1': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 7069.0}, 'accuracy': 0.8606709239987386, 'macro avg': {'precision': 0.4303354619993693, 'recall': 0.5, 'f1-score': 0.4625594525597703, 'support': 50736.0}, 'weighted avg': {'precision': 0.7407544394168424, 'recall': 0.8606709239987386, 'f1-score': 0.7962229428779364, 'support': 50736.0}}
FCNN evaluation with hyperparameter variations LR0.01 BS32: Time:0.02s, Memory:14.40MB
MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for tabular.
FCNN MC Dropout training with hyperparameter variations LR0.01 BS32: Ti

C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` par

 Early stopping at epoch 6
FCNN training completed.
Training model with seed 1
FCNN training started.
 Early stopping at epoch 6
FCNN training completed.
Training model with seed 2
FCNN training started.
 Early stopping at epoch 6
FCNN training completed.
Training model with seed 3
FCNN training started.
 Early stopping at epoch 6
FCNN training completed.
Training model with seed 4
FCNN training started.
 Early stopping at epoch 6
FCNN training completed.
Deep Ensemble finished for tabular.
FCNN Deep Ensemble training with hyperparameter variations LR0.01 BS32: Time:280.28s, Memory:3.89MB
Deep Ensembl uncertainty status:
Mean: 0.04675624519586563
Max: 0.11864632368087769
 90th percentile (threshold): 0.0302
Saved: results/figures\uncertainty_distribution_fcnn_training_with_hyperparameter_variations_lr0.01_bs32.png
SHAP started.


  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP with hyperparameter variations  BS32: Time:7.71s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME with hyperparameter variations LR0.01 BS32: Time:2.27s, Memory:13.23MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 25 Accuracy:64.0%
YELLOW: Count: 69 Accuracy:78.3%
GREEN: Count: 106 Accuracy:96.2%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.01_bs32.png
RED: Count: 32 Accuracy:75.0%
YELLOW: Count: 126 Accuracy:88.1%
GREEN: Count: 42 Accuracy:88.1%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.01_bs32.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 61 Accuracy:60.7%


...

SHAP finished. Explained: 200 samples.
FCNN SHAP with hyperparameter variations  BS128: Time:7.61s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME with hyperparameter variations LR0.01 BS128: Time:2.21s, Memory:12.00MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 18 Accuracy:61.1%
YELLOW: Count: 101 Accuracy:85.1%
GREEN: Count: 81 Accuracy:95.1%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.01_bs128.png
RED: Count: 33 Accuracy:72.7%
YELLOW: Count: 87 Accuracy:86.2%
GREEN: Count: 80 Accuracy:91.2%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.01_bs128.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 49 Accuracy:63.

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP with hyperparameter variations  BS32: Time:7.37s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME with hyperparameter variations LR0.0001 BS32: Time:2.70s, Memory:2.22MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 1 Accuracy:100.0%
YELLOW: Count: 43 Accuracy:83.7%
GREEN: Count: 156 Accuracy:87.2%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.0001_bs32.png
RED: Count: 3 Accuracy:100.0%
YELLOW: Count: 42 Accuracy:85.7%
GREEN: Count: 155 Accuracy:87.7%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.0001_bs32.png
MC Dropout without XAI Fl

  0%|          | 0/200 [00:00<?, ?it/s]

SHAP finished. Explained: 200 samples.
FCNN SHAP with hyperparameter variations  BS128: Time:7.48s, Memory:0.00MB
SHAP values shape: (200, 21, 2)
LIME started.
Explained 50 / 200 samples.
Explained 100 / 200 samples.
Explained 150 / 200 samples.
Explained 200 / 200 samples.
LIME finished. Explained: 200 samples.
FCNN LIME with hyperparameter variations LR0.0001 BS128: Time:1.86s, Memory:2.23MB
LIME explanations: 200
Calculate consistency tabular.
RED: Count: 2 Accuracy:100.0%
YELLOW: Count: 37 Accuracy:91.9%
GREEN: Count: 161 Accuracy:86.3%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.0001_bs128.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 37 Accuracy:91.9%
GREEN: Count: 163 Accuracy:85.9%
Flagging system is working
Saved: results/figures\flagging_distribution_fcnn_training_with_hyperparameter_variations_lr0.0001_bs128.png
MC Dropout without XAI Flagging:
RED: Count: 0 Accuracy